<a href="https://colab.research.google.com/github/dkoravski/ai-travel-planner/blob/main/AI_Travel_Planner_LangGraph_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Travel Planner — мулти-агентна система с LangGraph и LangChain
Пълно описание: виж README.md

In [ ]:
!pip install -q -U langgraph langchain langchain-openai langchain-community wikipedia


In [ ]:
import os
import uuid
import random
import time
import warnings
import requests
from typing import TypedDict, Annotated, Optional, List

# Потискаме безобидното предупреждение на bs4 ("GuessedAtParserWarning"), което
# библиотеката `wikipedia` (ползвана от wikipedia_search) поражда, защото не подава
# изрично features="lxml" на BeautifulSoup - не влияе на резултата, само на четимостта
# на изхода.
try:
    from bs4 import GuessedAtParserWarning
    warnings.filterwarnings("ignore", category=GuessedAtParserWarning)
except ImportError:
    pass

from langchain_core.messages import AnyMessage, HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_openai import ChatOpenAI

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_core.runnables.graph_mermaid import NodeStyles
from google.colab import userdata
from IPython.display import Image, display

model_name = "gpt-4o-mini"

In [ ]:
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] ="https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY1")
os.environ["LANGSMITH_PROJECT"] = "Travel Planner"

In [ ]:
def load_openai_key():
    # Зареждаме api_key през Google Colab Secrets
    try:
        api_key = userdata.get("OPENAI_API_KEY")
        if api_key:
            os.environ["OPENAI_API_KEY"] = api_key
            print("OpenAI API ключ зареден от Colab Secrets.")
            return
    except Exception:
        pass

load_openai_key()
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY не е зададен!"


In [ ]:
#State на графа (`TripState`)

class TripState(TypedDict):
    user_request: str                              # оригиналната заявка на потребителя
    thread_id: Optional[str]                        # id на нишката в checkpointer-а за тази заявка
    messages: Annotated[List[AnyMessage], add_messages]  # пълна история/памет на разговора
    research_notes: Optional[str]                  # резултат от Research Agent
    itinerary_draft: Optional[str]                  # текуща чернова на маршрута
    revision_count: int                             # брой направени ревизии
    human_feedback: Optional[str]                   # обратна връзка от човека (ако има)
    approved: Optional[bool]                        # дали човекът е одобрил чернова
    final_itinerary: Optional[str]                  # финалният, одобрен маршрут
    packing_list: Optional[str]                     # списък за багаж (генериран след одобрение)
    booking_confirmation: Optional[str]             # резултат от booking под-графа (резервации)
    needs_flight: Optional[bool]                    # преценка дали пътуването изисква самолетен билет
    flight_confirmed: Optional[bool]                # потвърждение от човека дали да се резервира полет


In [ ]:
#Системни класове и инструменти (Tools)

wikipedia_api = WikipediaAPIWrapper(top_k_results=2, doc_content_chars_max=1200)


@tool
def wikipedia_search(query: str) -> str:
    '''Search Wikipedia for background information about a travel destination (best season
    to visit, history, culture, notable facts). Wraps the built-in WikipediaAPIWrapper with
    a couple of retries and a graceful fallback message, so a transient Wikipedia/network
    error never crashes the whole multi-agent workflow.'''
    last_error = None
    for attempt in range(2):
        try:
            return wikipedia_api.run(query)
        except Exception as e:
            last_error = e
            time.sleep(1)
    return (
        f"Wikipedia търсенето за '{query}' не бе успешно в момента (временен проблем: "
        f"{type(last_error).__name__}: {last_error}). Продължи, разчитайки на общи познания "
        "за дестинацията, без да спираш работата."
    )


@tool
def search_flights_and_hotels(
    destination: str, num_days: int, num_travelers: int, currency_symbol: str = "$"
) -> str:
    '''Mock search for flight and hotel prices for a given destination, trip length (days)
    and number of travelers. Simulates a call to a real travel-booking API (e.g.
    Skyscanner/Booking.com) without needing a real key. IMPORTANT: pass currency_symbol
    matching whatever currency the user mentioned in their request (e.g. "$", "€", "£") -
    default is "$" (USD) if the user did not specify a currency.'''
    rng = random.Random(abs(hash(destination.lower().strip())) % (10**6))
    flight_low = rng.randint(250, 500)
    flight_high = flight_low + rng.randint(150, 400)
    hotel_per_night = rng.randint(60, 220)
    c = currency_symbol
    return (
        f"Дестинация: {destination}\n"
        f"Самолетни билети (round-trip, на пътник): {c}{flight_low}-{c}{flight_high}\n"
        f"Хотел (на нощувка, средна категория): {c}{hotel_per_night}\n"
        f"Прогнозна обща цена за настаняване ({num_days} нощувки): {c}{hotel_per_night * num_days}\n"
        f"Прогнозна обща цена за самолетни билети ({num_travelers} пътници): "
        f"{c}{flight_low * num_travelers}-{c}{flight_high * num_travelers}"
    )


WEATHER_CODE_MAP = {
    0: "слънчево", 1: "предимно слънчево", 2: "разкъсана облачност", 3: "облачно",
    45: "мъгливо", 48: "мъгливо",
    51: "лек дъжд", 53: "лек дъжд", 55: "дъжд", 56: "лек дъжд", 57: "дъжд",
    61: "лек дъжд", 63: "дъжд", 65: "дъжд", 66: "лек дъжд", 67: "дъжд",
    71: "лек снеговалеж", 73: "снеговалеж", 75: "снеговалеж", 77: "снеговалеж",
    80: "лек дъжд", 81: "дъжд", 82: "дъжд",
    85: "лек снеговалеж", 86: "снеговалеж",
    95: "гръмотевична буря", 96: "гръмотевична буря", 99: "гръмотевична буря",
}


def _mock_weather_forecast(destination: str, forecast_days: int) -> str:
    """Fallback генератор - ползва се само ако реалната Open-Meteo заявка се провали
    (мрежов проблем, дестинацията не е разпозната от geocoding-а и т.н.), за да не
    спира workflow-а."""
    conditions = [
        "слънчево", "предимно слънчево", "разкъсана облачност", "облачно",
        "лек дъжд", "дъжд", "гръмотевична буря", "мъгливо",
    ]
    rng = random.Random((abs(hash(destination.lower().strip())) % (10**6)) + 1)
    lines = [f"Прогноза за времето в {destination} (примерна, следващите {forecast_days} дни):"]
    for day in range(1, forecast_days + 1):
        condition = rng.choice(conditions)
        low = rng.randint(8, 20)
        high = low + rng.randint(3, 10)
        lines.append(f"  Ден {day}: {condition}, {low}°C - {high}°C")
    return "\n".join(lines)


@tool
def get_weather_forecast(destination: str, num_days: int) -> str:
    '''Real weather forecast tool for a travel destination, using the free Open-Meteo API
    (geocoding + daily forecast, no API key required). Covers up to the next 10 days,
    matching real-world forecast reliability limits (real weather services rarely forecast
    reliably much further ahead than ~10 days). Falls back to a generated mock forecast if
    the destination cannot be geocoded or the API call fails, so a transient network error
    never crashes the whole multi-agent workflow. Returns a day-by-day summary (condition +
    temperature range) that the Planner Agent can use to suggest indoor vs. outdoor
    activities in the itinerary.'''
    forecast_days = min(num_days, 10)
    try:
        geo_resp = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": destination, "count": 1, "language": "bg", "format": "json"},
            timeout=5,
        )
        geo_resp.raise_for_status()
        geo_results = geo_resp.json().get("results")
        if not geo_results:
            return _mock_weather_forecast(destination, forecast_days)

        lat = geo_results[0]["latitude"]
        lon = geo_results[0]["longitude"]

        forecast_resp = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": lat,
                "longitude": lon,
                "daily": "weathercode,temperature_2m_max,temperature_2m_min",
                "timezone": "auto",
                "forecast_days": forecast_days,
            },
            timeout=5,
        )
        forecast_resp.raise_for_status()
        daily = forecast_resp.json()["daily"]
        codes = daily["weathercode"]
        highs = daily["temperature_2m_max"]
        lows = daily["temperature_2m_min"]

        lines = [
            f"Прогноза за времето в {destination} (реални данни, Open-Meteo, следващите "
            f"{len(codes)} дни):"
        ]
        for i in range(len(codes)):
            condition = WEATHER_CODE_MAP.get(codes[i], "променливо")
            lines.append(f"  Ден {i + 1}: {condition}, {round(lows[i])}°C - {round(highs[i])}°C")
        if num_days > 10:
            lines.append(
                f"\nЗабележка: надеждна прогноза съществува само за първите {forecast_days} дни - "
                f"за останалите {num_days - forecast_days} дни от пътуването реалните прогнози "
                "не са достатъчно точни толкова напред във времето, затова не са включени тук."
            )
        return "\n".join(lines)
    except Exception:
        return _mock_weather_forecast(destination, forecast_days)


@tool
def estimate_daily_budget(
    total_budget: float, num_days: int, num_travelers: int, currency_symbol: str = "$"
) -> str:
    '''Calculator tool: splits a total trip budget across accommodation, food, activities
    and transport, per day and per traveler. IMPORTANT: pass currency_symbol matching
    whatever currency the user mentioned in their request (e.g. "$", "€", "£") - default is
    "$" (USD) if the user did not specify a currency.'''
    if num_days <= 0 or num_travelers <= 0:
        return "Грешка: брой дни и брой пътници трябва да са положителни числа."
    per_day_total = total_budget / num_days
    per_person_per_day = per_day_total / num_travelers
    breakdown = {
        "настаняване": 0.40,
        "храна": 0.25,
        "дейности/атракции": 0.20,
        "транспорт на място": 0.15,
    }
    c = currency_symbol
    lines = [
        f"Общ бюджет: {c}{total_budget:.2f} за {num_days} дни / {num_travelers} пътници",
        f"На ден (общо): {c}{per_day_total:.2f}",
        f"На пътник на ден: {c}{per_person_per_day:.2f}",
        "Препоръчана разбивка на ден:",
    ]
    for category, pct in breakdown.items():
        lines.append(f"  - {category}: {c}{per_day_total * pct:.2f} ({int(pct * 100)}%)")
    return "\n".join(lines)


RESEARCH_TOOLS = [wikipedia_search]
PLANNER_TOOLS = [search_flights_and_hotels, estimate_daily_budget, get_weather_forecast]

# Бърза проверка на custom инструментите (не изисква LLM/API ключ):
print(search_flights_and_hotels.invoke({"destination": "Lisabon", "num_days": 4, "num_travelers": 2, "currency_symbol": "€"}))
print()
print(estimate_daily_budget.invoke({"total_budget": 1200, "num_days": 4, "num_travelers": 2, "currency_symbol": "€"}))
print()
print(get_weather_forecast.invoke({"destination": "Lisabon", "num_days": 4}))

In [ ]:
#Агенти (system prompts + LLM)

RESEARCHER_SYSTEM_PROMPT = '''Ти си 'Destination Research Agent' - експерт по дестинации за пътуване.
Твоята единствена задача е да проучиш конкретна дестинация, като ЗАДЪЛЖИТЕЛНО използваш
инструмента за Wikipedia търсене, и да обобщиш кратко (в 5-8 изречения/точки):
- най-подходящ сезон/време за посещение,
- 3-5 задължителни забележителности/дейности,
- кратки културни/практически съвети (валута, език, транспорт).
Не съставяй маршрут по дни - това е задача на друг агент. Бъди конкретен и полезен.'''

PLANNER_SYSTEM_PROMPT = '''Ти си 'Itinerary Planner Agent' - експерт по съставяне на пътни маршрути.
Използваш инструментите за (1) търсене на цени на полети/хотели и (2) изчисляване на бюджет,
за да съставиш реалистичен, подробен маршрут по дни (Ден 1, Ден 2, ...), който пасва на бюджета
и предпочитанията на потребителя. Ако получиш обратна връзка от предишен преглед от човек,
ЗАДЪЛЖИТЕЛНО ревизирай маршрута така, че да я отразява. Форматирай маршрута ясно и четимо.

ВАЖНО за времето: ЗАДЪЛЖИТЕЛНО провери прогнозата за времето с наличния инструмент и съобрази
дневните дейности с нея - предлагай закрити дейности (музеи, ресторанти, спа) в дни с дъжд/лошо
време, и дейности на открито в слънчеви дни. Спомени накратко очакваното време за всеки ден
в текста на маршрута.

ВАЖНО за валутата: потребителят може да посочи бюджет в различна валута (напр. "$2000",
"€1500", "£1200"). Определи символа на валутата от заявката на потребителя и ЗАДЪЛЖИТЕЛНО
подавай точно този символ като параметър currency_symbol на двата инструмента (по подразбиране
е "$", ако потребителят не е посочил валута). Използвай СЪЩИЯ символ навсякъде в текста на
финалния маршрут, за да няма несъответствие между валутите.'''

llm = ChatOpenAI(model=model_name, temperature=0.4)

research_agent = create_agent(llm, tools=RESEARCH_TOOLS, system_prompt=RESEARCHER_SYSTEM_PROMPT)
planner_agent = create_agent(llm, tools=PLANNER_TOOLS, system_prompt=PLANNER_SYSTEM_PROMPT)


In [ ]:
#Nodes на графа

def research_node(state: TripState) -> dict:
    sub_result = research_agent.invoke({
        "messages": [HumanMessage(content=(
            f"Потребителска заявка: {state['user_request']}\n\n"
            "Проучи дестинацията, спазвайки инструкциите в системния промпт."
        ))]
    })
    notes = sub_result["messages"][-1].content
    return {
        "research_notes": notes,
        "messages": [AIMessage(content=f"[Research Agent]\n{notes}")],
    }


def planner_node(state: TripState) -> dict:
    feedback_block = ""
    if state.get("human_feedback"):
        feedback_block = (
            "\n\nОбратна връзка от потребителя за предходната чернова "
            f"(ЗАДЪЛЖИТЕЛНО я вземи предвид):\n{state['human_feedback']}"
        )
    prior_draft = ""
    if state.get("itinerary_draft"):
        prior_draft = f"\n\nПредходна чернова на маршрута:\n{state['itinerary_draft']}"

    prompt = (
        f"Заявка на потребителя: {state['user_request']}\n\n"
        f"Проучване за дестинацията:\n{state.get('research_notes', '(няма)')}"
        f"{prior_draft}{feedback_block}\n\n"
        "Изгради/ревизирай подробен дневен маршрут."
    )
    sub_result = planner_agent.invoke({"messages": [HumanMessage(content=prompt)]})
    draft = sub_result["messages"][-1].content
    new_revision_count = state.get("revision_count", 0) + (1 if state.get("human_feedback") else 0)
    return {
        "itinerary_draft": draft,
        "revision_count": new_revision_count,
        "human_feedback": None,
        "messages": [AIMessage(content=f"[Planner Agent - чернова №{new_revision_count + 1}]\n{draft}")],
    }


def human_gate_node(state: TripState) -> dict:
    # No-op "passthrough" възел. Паузата се случва защото графът е компилиран с
    # interrupt_before=["human_gate"]; докато графът е паузиран, външен код записва
    # 'approved'/'human_feedback' в state-а чрез update_state(), преди да продължи (resume).
    return {}


def finalize_node(state: TripState) -> dict:
    final_text = f"МАРШРУТЪТ Е ОДОБРЕН ОТ ПОТРЕБИТЕЛЯ.\n\n{state['itinerary_draft']}"
    return {
        "final_itinerary": final_text,
        "messages": [AIMessage(content=final_text)],
    }


def route_after_human_gate(state: TripState) -> str:
    return "finalize" if state.get("approved") else "planner_agent"


def packing_list_node(state: TripState) -> dict:
    """Прост node (без tools) - вика LLM-а директно с финалния одобрен маршрут, за да
    състави кратък списък за багаж. Демонстрира, че не всеки node трябва да е пълноценен
    ReAct агент - понякога един директен LLM-повик е достатъчен."""
    prompt = (
        "Въз основа на следната потребителска заявка и одобрен маршрут, състави кратък "
        "(8-12 точки) списък за багаж, съобразен с дестинацията, продължителността и "
        "евентуално споменатото време в маршрута. Форматирай като списък с тирета.\n\n"
        f"Заявка: {state['user_request']}\n\n"
        f"Одобрен маршрут:\n{state['final_itinerary']}"
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    packing_text = response.content
    return {
        "packing_list": packing_text,
        "messages": [AIMessage(content=f"[Packing List]\n{packing_text}")],
    }


def assess_booking_needs_node(state: TripState) -> dict:
    """Преценява дали пътуването изисква резервация на самолетен билет.

    Ако needs_flight вече е зададен ръчно отвън (виж force_needs_flight в
    execute_workflow) - използваме директно него и НЕ викаме LLM-а, за да можем
    детерминистично да демонстрираме и двата пътя (с/без полет) в тестовете."""
    if state.get("needs_flight") is not None:
        forced = state["needs_flight"]
        return {
            "messages": [AIMessage(
                content=f"[Booking Assessment] Нужен полет (зададено ръчно): {'да' if forced else 'не'}"
            )],
        }

    prompt = (
        "Прецени дали за следната заявка за пътуване е необходима резервация на самолетен "
        "билет, или дестинацията е достатъчно близка/вътрешна пътуването може да стане с "
        "кола/автобус/влак (в такъв случай НЕ е нужен полет). Отговори само с една дума: "
        "'да' (нужен е полет) или 'не' (не е нужен полет).\n\n"
        f"Заявка: {state['user_request']}"
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    answer = (response.content or "").strip().lower()
    needs_flight = answer.startswith("да")
    return {
        "needs_flight": needs_flight,
        "messages": [AIMessage(content=f"[Booking Assessment] Нужен полет: {'да' if needs_flight else 'не'}")],
    }


def prepare_booking_node(state: TripState) -> dict:
    """Подготвя ясна инструкция за booking_agent - вкарва в общата памет съобщение,
    указващо изрично дали е нужен полет, преди да преминем изпълнението към агента."""
    if state.get("needs_flight"):
        instruction = (
            f"Резервирай пътуването за заявката: {state['user_request']}\n"
            "Нужен е полет - извикай book_flight_tool И book_hotel_tool (по веднъж всеки)."
        )
    else:
        instruction = (
            f"Резервирай пътуването за заявката: {state['user_request']}\n"
            "НЕ е нужен полет - извикай САМО book_hotel_tool."
        )
    return {"messages": [HumanMessage(content=instruction)]}


@tool
def book_flight_tool(destination: str) -> str:
    """Резервира самолетен билет (отиване и връщане) до дадена дестинация. Извиквай
    само ако пътуването изисква полет."""
    rng = random.Random(abs(hash(destination.lower().strip())) % (10**6))
    outbound_code = f"FL-{rng.randint(1000, 9999)}"
    return_code = f"FL-{rng.randint(1000, 9999)}"
    return (
        f"Полет (отиване) резервиран успешно. Код: {outbound_code}.\n"
        f"Полет (връщане) резервиран успешно. Код: {return_code}."
    )


@tool
def book_hotel_tool(destination: str) -> str:
    """Резервира хотел за дадена дестинация."""
    rng = random.Random((abs(hash(destination.lower().strip())) % (10**6)) + 1)
    hotel_code = f"HT-{rng.randint(1000, 9999)}"
    return f"Хотел резервиран успешно. Код: {hotel_code}."


class BookingAgentState(AgentState):
    """Разширява стандартния AgentState (само `messages`) с полетата от TripState,
    които booking_agent трябва да чете/пише - тъй като е добавен directly като subgraph
    node (без адаптер функция), тези имена/типове трябва да съвпадат с TripState."""
    user_request: str
    needs_flight: Optional[bool]
    booking_confirmation: Optional[str]


BOOKING_SYSTEM_PROMPT = (
    "Ти си 'Booking агент' от мулти-агентна система за планиране на пътувания. "
    "Разполагаш с два инструмента: book_flight_tool (резервира самолетен билет) и "
    "book_hotel_tool (резервира хотел). Последното съобщение от потребителя ти казва "
    "изрично дали е нужен полет за това пътуване - следвай го точно: ако е нужен полет, "
    "извикай И ДВАТА инструмента; ако не е нужен - извикай само book_hotel_tool. Извиквай "
    "всеки инструмент точно веднъж. СЛЕД резултатите от инструментите отговори САМО с "
    "думата 'Готово.' - НЕ повтаряй кодовете или детайлите на резервацията, защото те вече "
    "се показват отделно на потребителя (в отговора на инструментите и във финалното "
    "потвърждение)."
)

# HumanInTheLoopMiddleware спира агента точно преди изпълнение на book_flight_tool
# (истинска резервация на самолетен билет), за да поиска потвърждение от човек - това е
# HITL механизъм, вграден в самия агент (ниско ниво, виж README за сравнение с
# interrupt_before, ползван за human_gate).
booking_agent = create_agent(
    model=llm,
    tools=[book_flight_tool, book_hotel_tool],
    state_schema=BookingAgentState,
    system_prompt=BOOKING_SYSTEM_PROMPT,
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"book_flight_tool": True})],
)
# ВАЖНО: booking_agent се добавя directly като node в graph_builder (виж по-долу) - НЕ е
# увит в обикновена функция. Само така LangGraph пази checkpoint-ите на прекъсването
# вътре в агента като част от главния граф, а resume-ването продължава от точното място.


def finalize_booking_node(state: TripState) -> dict:
    """Извлича резултатите от book_flight_tool/book_hotel_tool (ToolMessage-и, записани
    от booking_agent в общата памет) и съставя финално обобщено потвърждение.

    Важно: дали е бил резервиран полет се определя от РЕАЛНИЯ изход на book_flight_tool
    (успешен ToolMessage), а не само от предварителната преценка needs_flight - ако
    човекът отхвърли резервацията на полет през HITL прекъсването, tool-ът изобщо не се
    изпълнява и потвърждението не бива да твърди, че полет е резервиран."""
    tool_msgs = [
        m for m in state["messages"]
        if isinstance(m, ToolMessage) and m.name in ("book_flight_tool", "book_hotel_tool")
    ]
    flight_booked = any(
        m.name == "book_flight_tool" and getattr(m, "status", "success") != "error"
        for m in tool_msgs
    )
    prev = "\n".join(str(m.content) for m in tool_msgs)
    what_was_booked = "полет + хотел" if flight_booked else "само хотел (без полет)"
    final_text = (
        f"{prev}\n\n✅ ПОТВЪРЖДЕНИЕ: Цялото пътуване ({what_was_booked}) е резервирано "
        "успешно. Приятно пътуване!"
    )
    return {
        "booking_confirmation": final_text,
        "flight_confirmed": flight_booked,
        "messages": [AIMessage(content=f"[Booking] {final_text}")],
    }

## Изграждане на графа (памет + HITL прекъсване)

In [ ]:
checkpointer = MemorySaver()

In [ ]:
graph_builder = StateGraph(TripState)
graph_builder.add_node("research_agent", research_node)
graph_builder.add_node("planner_agent", planner_node)
graph_builder.add_node("human_gate", human_gate_node)
graph_builder.add_node("finalize", finalize_node)
graph_builder.add_node("packing_list", packing_list_node)
graph_builder.add_node("assess_booking_needs", assess_booking_needs_node)
graph_builder.add_node("prepare_booking", prepare_booking_node)
graph_builder.add_node("booking_agent", booking_agent)
graph_builder.add_node("finalize_booking", finalize_booking_node)

graph_builder.add_edge(START, "research_agent")
graph_builder.add_edge("research_agent", "planner_agent")
graph_builder.add_edge("planner_agent", "human_gate")
graph_builder.add_conditional_edges(
    "human_gate",
    route_after_human_gate,
    {"finalize": "finalize", "planner_agent": "planner_agent"},
)
graph_builder.add_edge("finalize", "packing_list")
graph_builder.add_edge("packing_list", "assess_booking_needs")
graph_builder.add_edge("assess_booking_needs", "prepare_booking")
graph_builder.add_edge("prepare_booking", "booking_agent")
graph_builder.add_edge("booking_agent", "finalize_booking")
graph_builder.add_edge("finalize_booking", END)

# Забележка: тук имаме само ЕДНА статична interrupt точка (human_gate,
# interrupt_before). Втората HITL точка - потвърждение на резервацията на полет - е
# динамичен interrupt() извикан отвътре в booking_agent чрез
# HumanInTheLoopMiddleware.
travel_graph = graph_builder.compile(
    checkpointer=checkpointer, interrupt_before=["human_gate"]
)

# display(Image(travel_graph.get_graph().draw_mermaid_png()))
display(Image(travel_graph.get_graph(xray=True).draw_mermaid_png(
    node_colors=NodeStyles(
        first="fill:#bbf7d0,stroke:#16a34a,stroke-width:2px",   # Start - видим зелен овал
        last="fill:#fca5a5,stroke:#dc2626,stroke-width:2px",    # End - видим червен овал
    )
)))


In [ ]:
#Функция execute_workflow

APPROVE_KEYWORDS = {"одобрявам", "одобрено", "approve", "approved", "yes", "да", "ok", "ок", "потвърждавам"}


def default_human_input(itinerary_draft: str) -> str:
    print("\n" + "=" * 70)
    print("МАРШРУТ ЗА ЧОВЕШКИ ПРЕГЛЕД:\n")
    print(itinerary_draft)
    print("=" * 70)
    return input("\nОдобрявате ли? Напишете 'одобрявам' или опишете какво да се промени: ")


def execute_workflow(
    user_request: str,
    human_input_fn=None,
    max_revisions: int = 3,
    verbose: bool = True,
    force_needs_flight: Optional[bool] = None,
) -> dict:
    '''Основна входна функция на мулти-агентния workflow.

    Параметри
    ----------
    user_request : str
        Заявката на потребителя на естествен език, напр.
        "Планирай 5-дневно пътуване до Токио с бюджет €2000 за 2-ма души."
    human_input_fn : callable, optional
        Функция (itinerary_draft: str) -> str, връщаща отговора на човека при преглед.
        По подразбиране използва истинско input() (интерактивен режим). Тестовите кейси
        подават функция със симулирани отговори, за да демонстрират approve/revise потока
        детерминистично, без да чакат реално въвеждане.
    max_revisions : int
        Предпазен таван на броя ревизии (за да не се получи безкраен цикъл).
    force_needs_flight : bool, optional
        Ако е зададен (True/False), пропуска LLM преценката в assess_booking_needs и
        директно налага дали е нужен полет - полезно за детерминистично демонстриране
        и на двата пътя (с/без резервация на полет) в тестовете. По подразбиране (None)
        решението се взима автоматично от LLM-а според заявката.

    Връща: финалното state на графа (dict), включващо 'final_itinerary'.
    '''
    if human_input_fn is None:
        human_input_fn = default_human_input

    thread_id = str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id}}

    initial_state = {
        "user_request": user_request,
        "thread_id": thread_id,
        "messages": [HumanMessage(content=user_request)],
        "revision_count": 0,
        "human_feedback": None,
        "approved": None,
        "needs_flight": force_needs_flight,
        "flight_confirmed": None,
    }

    if verbose:
        print(f"\n🧭  Стартиране на workflow за заявка: {user_request!r}")

    state = travel_graph.invoke(initial_state, config)

    revisions = 0
    while True:
        snapshot = travel_graph.get_state(config)
        next_nodes = snapshot.next

        if not next_nodes:
            break

        if "human_gate" in next_nodes:
            response = human_input_fn(state["itinerary_draft"])
            approved = response.strip().lower() in APPROVE_KEYWORDS

            if verbose:
                verdict = "ОДОБРЕНО ✅" if approved else "ИСКА СЕ РЕВИЗИЯ 🔁"
                print(f"\n👤 Отговор на потребителя: {response!r}  ->  {verdict}")

            travel_graph.update_state(config, {
                "approved": approved,
                "human_feedback": None if approved else response,
            })
            state = travel_graph.invoke(None, config)

            if not approved:
                revisions += 1
                if revisions >= max_revisions:
                    if verbose:
                        print("⚠️  Достигнат е максималният брой ревизии.")
                    break
            continue

        if snapshot.interrupts:
            # Динамичен interrupt() отвътре в booking_agent (HumanInTheLoopMiddleware),
            # преди изпълнение на book_flight_tool - различен resume механизъм от
            # human_gate: Command(resume=...) вместо update_state()+invoke(None, ...).
            hitl_request = snapshot.interrupts[0].value
            num_pending = len(hitl_request["action_requests"])

            question = (
                "Booking агентът иска да резервира самолетен билет - да продължим ли? "
                "Напишете 'одобрявам' (да) или друг отговор (без резервация на полет)."
            )
            response = human_input_fn(question)
            flight_confirmed = response.strip().lower() in APPROVE_KEYWORDS

            if verbose:
                verdict = "РЕЗЕРВИРАМЕ ПОЛЕТ ✈️" if flight_confirmed else "ОТКАЗ НА ПОЛЕТ 🚗"
                print(f"\n👤 Отговор на потребителя: {response!r}  ->  {verdict}")

            decision = (
                {"type": "approve"}
                if flight_confirmed
                else {"type": "reject", "message": "Потребителят отказа резервацията на полет."}
            )
            state = travel_graph.invoke(
                Command(resume={"decisions": [decision] * num_pending}), config
            )
            continue

        break

    if verbose:
        print("\n✅ Workflow завършен.\n")
    return state


In [ ]:
def simulated_reviewer(*responses):
    '''Връща функция, която на всяко повикване дава следващия отговор от 'responses'
    (симулира човек, отговарящ на последователни прегледи на маршрута/резервацията).
    Ако бъде повикана повече пъти, отколкото са зададените отговори (напр. заради
    допълнителния въпрос за резервация на полет от booking_flight_gate), повтаря последния
    зададен отговор, вместо да гръмне с грешка.'''
    responses = list(responses) or ["одобрявам"]
    it = iter(responses)
    def _fn(itinerary_draft: str) -> str:
        nonlocal it
        try:
            answer = next(it)
        except StopIteration:
            answer = responses[-1]
        print("\n" + "=" * 70)
        print("ЗА ЧОВЕШКИ ПРЕГЛЕД:\n")
        print(itinerary_draft)
        print("=" * 70)
        print(f"[СИМУЛИРАН ЧОВЕШКИ ОТГОВОР] -> {answer!r}")
        return answer
    return _fn


def print_memory_transcript(result_state: dict) -> None:
    print("\n--- ПЪЛНА ПАМЕТ НА РАЗГОВОРА (messages) ---")
    for m in result_state["messages"]:
        role = m.__class__.__name__
        preview = m.content if len(m.content) < 300 else m.content[:300] + " ..."
        print(f"[{role}] {preview}\n")


## Тестови сценарии (5 теста)

In [ ]:
# Тест 1: Токио, одобрение веднага ---
result_1 = execute_workflow(
    "Планирай 5-дневно пътуване до Токио, Япония, за 2-ма възрастни, с бюджет €2000. "
    "Искаме комбинация от култура и модерен градски живот.",
    human_input_fn=simulated_reviewer("одобрявам", "одобрявам"),  # 2-ри отговор = потвърждение за полет
    force_needs_flight=True,  # демонстративно налагаме, че за Токио е нужен полет
)
print(result_1["final_itinerary"])
print_memory_transcript(result_1)


In [ ]:
#Демонстрация: packing_list и booking

print("--- Списък за багаж (Тест 1) ---")
print(result_1["packing_list"])
print()
print("--- Потвърждение за резервация (Тест 1) ---")
print(result_1["booking_confirmation"])


In [ ]:
# Тест 2: Париж, обратна връзка -> ревизия -> одобрение ---
result_2 = execute_workflow(
    "Искам романтичен уикенд (3 дни) в Париж, Франция, за 2-ма, бюджет €1500.",
    human_input_fn=simulated_reviewer(
        "Маршрутът е твърде натоварен - искам повече свободно време и по-малко музеи.",
        "одобрявам",
    ),
)
print(result_2["final_itinerary"])
print_memory_transcript(result_2)


In [ ]:
# Тест 3: Родопите, 2-ма, обратна връзка за бюджета -> ревизия -> одобрение ---
result_3 = execute_workflow(
    "Планирай 3-дневно семейно пътуване до Родопите, България, за 2-ма възрастни, "
    "бюджет €600.",
    human_input_fn=simulated_reviewer(
        "Настаняването изглежда твърде скъпо за нашия бюджет - предложи по-икономичен вариант.",
        "одобрявам",
    ),
)
print(result_3["final_itinerary"])
print_memory_transcript(result_3)


In [ ]:
#Демонстрация на nodes packing_list и booking

print("--- Списък за багаж (Тест 3) ---")
print(result_3["packing_list"])
print()
print("--- Потвърждение за резервация (Тест 3) ---")
print(result_3["booking_confirmation"])

In [ ]:
# Тест 4: Бали, меден месец, одобрение веднага ---
result_4 = execute_workflow(
    "Планирай 7-дневен меден месец на остров Бали, Индонезия, за 2-ма, бюджет €4000, "
    "искаме плажове, спа и малко приключения.",
    human_input_fn=simulated_reviewer("одобрявам"),
)
print(result_4["final_itinerary"])
print_memory_transcript(result_4)


In [ ]:
# Тест 5: Ню Йорк, бизнес + свободно време, ДВЕ последователни ревизии -> одобрение ---
result_5 = execute_workflow(
    "Планирай 4-дневно пътуване до Ню Йорк, САЩ, за 1 човек (бизнес пътуване с малко "
    "свободно време), бюджет €1800.",
    human_input_fn=simulated_reviewer(
        "Добави повече препоръки за музеи и култура през свободното време.",
        "Все още искам поне един ден изцяло свободен, без планирани активности.",
        "одобрявам",
    ),
)
print(result_5["final_itinerary"])
print(f"\nОбщ брой ревизии за този тест: {result_5['revision_count']}")
print_memory_transcript(result_5)


In [ ]:
config_1 = {"configurable": {"thread_id": result_1["thread_id"]}}

for checkpoint_tuple in checkpointer.list(config_1):
    is_interrupt = (
        checkpoint_tuple.config["configurable"]["checkpoint_ns"] == ""
        and checkpoint_tuple.metadata.get("writes") is None
    )
    marker = "  ⏸️ interrupt (графът е спрян тук)" if is_interrupt else ""
    print(checkpoint_tuple, marker)
    print("-" * 100)

In [ ]:
#Интерактивен режим (реален input())
# Изисква реално въвеждане в Colab:
my_result = execute_workflow("Планирай 4-дневно пътуване до Виена, Австрия, за 2-ма, бюджет €1600.")
print(my_result["final_itinerary"])
